### **Explore data in a DataFrame**

In [1]:
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType

# define the schema
schema = StructType() \
.add("ProductID", IntegerType(), True) \
.add("ProductName", StringType(), True) \
.add("Category", StringType(), True) \
.add("ListPrice", DoubleType(), True)

df = spark.read.format("csv").option("header","true").schema(schema).load("Files/products/products.csv")
# df now is a Spark DataFrame containing CSV data from "Files/products/products.csv".
display(df)

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ead6ecde-17c3-4ed2-aac1-7717440990a1)

### **Create a Delta table**

In [2]:
df.write.format("delta").saveAsTable("dbo.products_table")

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 4, Finished, Available, Finished, False)

### **Explore table versioning**

In [3]:
%%sql
UPDATE dbo.products_table
SET ListPrice = ListPrice * 0.9
WHERE Category = 'Mountain Bikes';

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [4]:
%%sql
DESCRIBE HISTORY dbo.products_table;

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 6, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 15 fields>

In [5]:
%%sql
SELECT
    o.ProductName,
    o.ListPrice AS OriginalPrice,
    u.ListPrice AS UpdatedPrice
FROM dbo.products_table VERSION AS OF 0 o
JOIN dbo.products_table u ON o.ProductID = u.ProductID
WHERE o.Category = 'Mountain Bikes'
ORDER BY o.ProductName;

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 7, Finished, Available, Finished, False)

<Spark SQL result set with 32 rows and 3 fields>

### **Analyze Delta table data with SQL queries**

In [12]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW products_view
AS
    SELECT Category, COUNT(*) AS NumProducts, MIN(ListPrice) AS MinPrice, MAX(ListPrice) AS MaxPrice, AVG(ListPrice) AS AvgPrice
    FROM `Hands-On`.delta_lakehouse.dbo.products_table
    GROUP BY Category;

SELECT *
FROM products_view
ORDER BY Category;

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 15, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 37 rows and 5 fields>

In [13]:
%%sql
SELECT Category, NumProducts
FROM products_view
ORDER BY NumProducts DESC
LIMIT 10;

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 16, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 2 fields>

In [14]:
from pyspark.sql.functions import col, desc

df_products = spark.sql("SELECT Category, MinPrice, MaxPrice, AvgPrice FROM products_view").orderBy(col("AvgPrice").desc())
display(df_products.limit(6))

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a5eca812-7a38-408c-acdf-84601a7cb4f9)

### **Use Delta tables for streaming data**

In [15]:
from notebookutils import mssparkutils
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Create a folder
inputPath = 'Files/data/'
mssparkutils.fs.mkdirs(inputPath)

# Create a stream that reads data from the folder, using a JSON schema
jsonSchema = StructType([
StructField("device", StringType(), False),
StructField("status", StringType(), False)
])
iotstream = spark.readStream.schema(jsonSchema).option("maxFilesPerTrigger", 1).json(inputPath)

# Write some event data to the folder
device_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "data.txt", device_data, True)

print("Source stream created...")

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 18, Finished, Available, Finished, False)

Source stream created...


In [16]:
# Write the stream to a delta table
delta_stream_table_path = 'Tables/dbo/iotdevicedata'
checkpointpath = 'Files/delta/checkpoint'
deltastream = iotstream.writeStream.format("delta").option("checkpointLocation", checkpointpath).start(delta_stream_table_path)
print("Streaming to delta sink...")

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 19, Finished, Available, Finished, False)

Streaming to delta sink...


In [17]:
%%sql
SELECT * FROM dbo.IotDeviceData;

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 20, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 2 fields>

In [18]:
# Add more data to the source stream
more_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "more-data.txt", more_data, True)

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 21, Finished, Available, Finished, False)

True

In [19]:
%%sql
SELECT * FROM dbo.IotDeviceData;

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 22, Finished, Available, Finished, False)

<Spark SQL result set with 16 rows and 2 fields>

In [20]:
deltastream.stop()

StatementMeta(, 3688790f-bc2a-4eae-9b94-fb206960b3a0, 23, Finished, Available, Finished, False)